In [58]:
import pandas as pd
import numpy as np
import glob
from pathlib import Path
import matplotlib.pyplot as plt
import seaborn as sns
import re
import io

In [59]:
data_root_path = Path('/astrum/home/hpchzy/code/data/20260623/cgnr6760pn2/output_polybench_gemm_filt_dsub/20260623-full-v2-polybench-gemm-dsub')

In [66]:
df_tot = []
for csv_file in data_root_path.rglob('shuffle[0-9]/*gemm*/**/*.csv'):
    shuffle = re.search(r'shuffle(?P<shuffle>[0-9])', str(csv_file)).group('shuffle')
    if re.search(r'.*(filt|tf)$', csv_file.parent.name):
        continue
    # print(csv_file.parent.name)
    parent_groups = re.search(r'.*?gemm_(?P<timer>.*?)_np(?P<np>\d+)_size(?P<size>\d+)$', csv_file.parent.name)
    # if parent_groups is not None:
    #     print(parent_groups.groupdict())
    csv_groups = re.search(r'.*?gemm_(?P<timer>.*?)_(?P<rank>[0-9]+)_(?P<host>.*).csv', csv_file.name)
    # if csv_groups is not None:
        # print(csv_groups.groupdict())
    meta_dict= {**csv_groups.groupdict(), **parent_groups.groupdict()}
    # print(meta_dict)
    df = pd.read_csv(csv_file,header=None, names=['ns'])
    ns_np = np.array(df['ns'], dtype=np.int64)
    meta_dict['ns'] = ns_np
    meta_dict['shuffle'] = shuffle
    df_tot.append(meta_dict)

df_tot = pd.DataFrame(df_tot)
df_tot.head()

,timer,rank,host,np,size,ns,shuffle
0,papi,21,cgnr6760pn2,64,32,"[778, 777, 780, 778, 780, 777, 778, 777, 780, ...",0
1,papi,63,cgnr6760pn2,64,32,"[775, 774, 780, 775, 782, 783, 781, 775, 779, ...",0
2,papi,4,cgnr6760pn2,64,32,"[776, 777, 777, 780, 778, 776, 779, 776, 775, ...",0
3,papi,42,cgnr6760pn2,64,32,"[779, 779, 778, 782, 776, 774, 777, 775, 774, ...",0
4,papi,31,cgnr6760pn2,64,32,"[781, 778, 779, 781, 779, 777, 776, 782, 780, ...",0


In [69]:
df_tot['mean'] = df_tot['ns'].apply(np.mean)
df_tot['q25'] = df_tot['ns'].apply(lambda x: np.percentile(x, 25))
df_tot['q50'] = df_tot['ns'].apply(lambda x: np.percentile(x, 50))
df_tot['q75'] = df_tot['ns'].apply(lambda x: np.percentile(x, 75))

df_tot.head()

,timer,rank,host,np,size,ns,shuffle,mean,q25,q50,q75
0,papi,21,cgnr6760pn2,64,32,"[778, 777, 780, 778, 780, 777, 778, 777, 780, ...",0,792.812500,775.0,777.0,779.0
1,papi,63,cgnr6760pn2,64,32,"[775, 774, 780, 775, 782, 783, 781, 775, 779, ...",0,800.734375,778.0,781.0,797.0
2,papi,4,cgnr6760pn2,64,32,"[776, 777, 777, 780, 778, 776, 779, 776, 775, ...",0,777.293750,776.0,777.0,779.0
3,papi,42,cgnr6760pn2,64,32,"[779, 779, 778, 782, 776, 774, 777, 775, 774, ...",0,777.628125,776.0,778.0,780.0
4,papi,31,cgnr6760pn2,64,32,"[781, 778, 779, 781, 779, 777, 776, 782, 780, ...",0,777.590625,776.0,777.0,779.0
